In [1]:
# ============================================================================
# Swin Transformer + Temporal Attention — Riverbank Prediction (YEARLY)
# ST-Transformer: Hierarchical Spatial Encoding + Cross-Temporal Attention
# Multiple Sequence Lengths = [4, 5, 6] → 200 Epochs + Early Stopping
# STRICT Temporal Split (Triple-Layer Leak Prevention)
# ============================================================================

# ── Suppress warnings BEFORE importing tensorflow ──
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['XLA_FLAGS'] = '--xla_gpu_strict_conv_algorithm_picker=false'
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'

import re
import glob
import sys
import logging
import numpy as np
import pandas as pd

logging.getLogger('tensorflow').setLevel(logging.FATAL)
logging.getLogger('absl').setLevel(logging.FATAL)

import tensorflow as tf
tf.get_logger().setLevel('FATAL')
tf.autograph.set_verbosity(0)

import warnings
warnings.filterwarnings('ignore')
import absl.logging
absl.logging.set_verbosity(absl.logging.FATAL)
absl.logging._warn_preinit_stderr = False

try:
    import keras
    keras.config.enable_unsafe_deserialization()
except AttributeError:
    try:
        tf.keras.config.enable_unsafe_deserialization()
    except AttributeError:
        pass

from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
)
from tensorflow.keras.optimizers import Adam
import rasterio
from sklearn.metrics import precision_score, recall_score
from skimage.transform import resize
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

np.random.seed(42)
tf.random.set_seed(42)

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU configured: {len(gpus)} GPU(s) available")
    except RuntimeError as e:
        print(f"GPU Error: {e}")
else:
    print("⚠ No GPU found, using CPU")

_warmup = tf.zeros((1, 1), dtype=tf.float32)
_ = tf.matmul(_warmup, _warmup)
del _warmup

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_DIR = "/kaggle/input/datasets/kaoserahamed/riverbank/river"
IMG_HEIGHT = 256
IMG_WIDTH = 256
SEQUENCE_LENGTHS = [4, 5, 6]
PREDICTION_HORIZON = 1
STRIDE = 1
EPOCHS = 200
EARLY_STOP_PATIENCE = 20
REDUCE_LR_PATIENCE = 7
BATCH_SIZE = 2
N_COMPONENTS = 3

# ── ST-Transformer Hyperparameters ──
PATCH_SIZE = 4
EMBED_DIM = 64
SWIN_DEPTHS = [2, 2, 2]
SWIN_NUM_HEADS = [2, 4, 8]
WINDOW_SIZE = 8
MLP_RATIO = 2.0
SPATIAL_DROPOUT = 0.1

NUM_TEMPORAL_LAYERS = 2
NUM_TEMPORAL_HEADS = 4
TEMPORAL_MLP_DIM = 256
TEMPORAL_DROPOUT = 0.1

DECODER_FILTERS = [128, 64, 32, 16]

SETUP_CONFIGS = {
    'Setup 1': {'cutoff_year': 2015, 'test_label': 'Test: 2016-2025'},
    'Setup 2': {'cutoff_year': 2020, 'test_label': 'Test: 2021-2025'},
}

# ============================================================================
# PREPROCESSING FUNCTIONS
# ============================================================================

def keep_largest_n_components_cv2(mask, n=1):
    binary_mask = (mask > 0).astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )
    if num_labels <= 1:
        return np.zeros_like(mask)
    areas = stats[1:, cv2.CC_STAT_AREA]
    sorted_indices = np.argsort(areas)[::-1]
    n_to_keep = min(n, len(sorted_indices))
    top_n_indices = sorted_indices[:n_to_keep]
    cleaned_mask = np.zeros_like(mask)
    for idx in top_n_indices:
        actual_label = idx + 1
        cleaned_mask[labels == actual_label] = mask.max()
    return cleaned_mask


def load_and_preprocess_image(filepath, apply_cleaning=True,
                              n_components=N_COMPONENTS):
    with rasterio.open(filepath) as src:
        img = src.read(1)
    if apply_cleaning:
        img = keep_largest_n_components_cv2(img, n=n_components)
    img_resized = resize(
        img, (IMG_HEIGHT, IMG_WIDTH),
        mode='constant', preserve_range=True, anti_aliasing=False
    )
    img_binary = (img_resized > 0.5).astype(np.float32)
    return img_binary


def create_sequences_with_years(images, years, seq_len):
    X, y = [], []
    input_years_list, target_years_list = [], []
    input_idx_list, target_idx_list = [], []
    for i in range(0, len(images) - seq_len - PREDICTION_HORIZON + 1, STRIDE):
        X.append(images[i:i + seq_len])
        y.append(images[i + seq_len])
        input_years_list.append(years[i:i + seq_len])
        target_years_list.append(years[i + seq_len])
        input_idx_list.append(list(range(i, i + seq_len)))
        target_idx_list.append(i + seq_len)
    return (np.array(X), np.array(y),
            np.array(input_years_list), np.array(target_years_list),
            input_idx_list, np.array(target_idx_list))


# ============================================================================
# LOSSES AND METRICS
# ============================================================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        K.sum(y_true_f) + K.sum(y_pred_f) + smooth
    )


def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)


def combined_loss(y_true, y_pred):
    bce = K.mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)


def iou_metric(y_true, y_pred, threshold=0.5):
    y_pred_binary = K.cast(y_pred > threshold, dtype='float32')
    y_true_binary = K.cast(y_true > threshold, dtype='float32')
    intersection = K.sum(y_true_binary * y_pred_binary)
    union = K.sum(y_true_binary) + K.sum(y_pred_binary) - intersection
    return (intersection + K.epsilon()) / (union + K.epsilon())


def dice_coefficient_np(y_true, y_pred, threshold=0.5):
    y_true_b = (y_true > threshold).astype(np.float32)
    y_pred_b = (y_pred > threshold).astype(np.float32)
    intersection = np.sum(y_true_b * y_pred_b)
    return (2. * intersection) / (np.sum(y_true_b) + np.sum(y_pred_b) + 1e-6)


def iou_np(y_true, y_pred, threshold=0.5):
    y_true_b = (y_true > threshold).astype(np.float32)
    y_pred_b = (y_pred > threshold).astype(np.float32)
    intersection = np.sum(y_true_b * y_pred_b)
    union = np.sum(y_true_b) + np.sum(y_pred_b) - intersection
    return intersection / (union + 1e-6)


def calculate_area_difference(y_true, y_pred, pixel_area_km2, threshold=0.5):
    true_area = np.sum((y_true > threshold).astype(np.float32)) * pixel_area_km2
    pred_area = np.sum((y_pred > threshold).astype(np.float32)) * pixel_area_km2
    return pred_area - true_area


# ============================================================================
# TRAINING CALLBACK
# ============================================================================

class StructuredTrainingLogger(Callback):
    def __init__(self, total_epochs):
        super().__init__()
        self.total_epochs = total_epochs
        self.best_val_loss = np.inf

    def on_train_begin(self, logs=None):
        header = (
            f"{'Ep':>4s}/{'Tot':<4s} │ {'Loss':>8s} │ {'VLoss':>8s} │ "
            f"{'Dice':>6s} │ {'VDice':>6s} │ {'IoU':>6s} │ "
            f"{'VIoU':>6s} │ {'LR':>9s} │ Note"
        )
        print("─" * len(header))
        print(header)
        print("─" * len(header))

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = logs.get('loss', 0)
        vloss = logs.get('val_loss', 0)
        dice = logs.get('dice_coefficient', 0)
        vdice = logs.get('val_dice_coefficient', 0)
        iou = logs.get('iou_metric', 0)
        viou = logs.get('val_iou_metric', 0)
        lr = float(K.get_value(self.model.optimizer.learning_rate))
        note = ""
        if vloss < self.best_val_loss:
            self.best_val_loss = vloss
            note = "★ saved"
        print(
            f"{epoch+1:>4d}/{self.total_epochs:<4d} │ {loss:>8.4f} │ "
            f"{vloss:>8.4f} │ {dice:>6.4f} │ {vdice:>6.4f} │ "
            f"{iou:>6.4f} │ {viou:>6.4f} │ {lr:>9.2e} │ {note}"
        )

    def on_train_end(self, logs=None):
        print("─" * 85)
        print(f"  Training finished  │  Best val_loss: {self.best_val_loss:.4f}")
        print("─" * 85)


def create_callbacks(model_name, checkpoint_dir='checkpoints'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    ckpt_path = os.path.join(checkpoint_dir, f'{model_name}_best.keras')
    return [
        ModelCheckpoint(ckpt_path, monitor='val_loss', save_best_only=True,
                        save_weights_only=False, mode='min', verbose=0),
        EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=REDUCE_LR_PATIENCE, min_lr=1e-7, verbose=0),
        StructuredTrainingLogger(total_epochs=EPOCHS),
    ]


# ============================================================================
# ST-TRANSFORMER CUSTOM LAYERS
# ============================================================================

class PatchEmbedSwin(layers.Layer):
    def __init__(self, patch_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.proj = layers.Conv2D(
            embed_dim, kernel_size=patch_size, strides=patch_size, padding='valid'
        )
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x):
        x = self.proj(x)
        x = self.norm(x)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({'patch_size': self.patch_size, 'embed_dim': self.embed_dim})
        return config


class PatchMerging(layers.Layer):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.output_dim = output_dim
        self.reduction = layers.Conv2D(
            output_dim, kernel_size=2, strides=2, padding='valid'
        )
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x):
        x = self.reduction(x)
        x = self.norm(x)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({'output_dim': self.output_dim})
        return config


class WindowPartition(layers.Layer):
    def __init__(self, window_size, **kwargs):
        super().__init__(**kwargs)
        self.window_size = window_size

    def call(self, x):
        x_shape = tf.shape(x)
        B, H, W, C = x_shape[0], x_shape[1], x_shape[2], x_shape[3]
        ws = self.window_size
        nH = H // ws
        nW = W // ws
        x = tf.reshape(x, [B, nH, ws, nW, ws, C])
        x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
        x = tf.reshape(x, [B * nH * nW, ws, ws, C])
        return x

    def get_config(self):
        config = super().get_config()
        config.update({'window_size': self.window_size})
        return config


class WindowReverse(layers.Layer):
    def __init__(self, window_size, h_patches, w_patches, **kwargs):
        super().__init__(**kwargs)
        self.window_size = window_size
        self.h_patches = h_patches
        self.w_patches = w_patches

    def call(self, windows):
        ws = self.window_size
        nH = self.h_patches // ws
        nW = self.w_patches // ws
        C = tf.shape(windows)[-1]
        B = tf.shape(windows)[0] // (nH * nW)
        x = tf.reshape(windows, [B, nH, nW, ws, ws, C])
        x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
        x = tf.reshape(x, [B, self.h_patches, self.w_patches, C])
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'window_size': self.window_size,
            'h_patches': self.h_patches,
            'w_patches': self.w_patches,
        })
        return config


class WindowAttentionBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, window_size, h_patches, w_patches,
                 mlp_ratio=2.0, dropout_rate=0.1, shift=False, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.h_patches = h_patches
        self.w_patches = w_patches
        self.mlp_ratio = mlp_ratio
        self.dropout_rate = dropout_rate
        self.shift = shift
        self.shift_size = window_size // 2 if shift else 0

    def build(self, input_shape):
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.window_partition = WindowPartition(self.window_size)
        self.window_reverse = WindowReverse(
            self.window_size, self.h_patches, self.w_patches
        )
        ws_sq = self.window_size * self.window_size
        self.reshape_to_seq = layers.Reshape((ws_sq, self.embed_dim))
        self.attn = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.embed_dim // self.num_heads,
            dropout=self.dropout_rate
        )
        self.reshape_to_2d = layers.Reshape(
            (self.window_size, self.window_size, self.embed_dim)
        )
        self.drop1 = layers.Dropout(self.dropout_rate)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        mlp_hidden = int(self.embed_dim * self.mlp_ratio)
        self.mlp_dense1 = layers.Dense(mlp_hidden, activation='gelu')
        self.mlp_drop = layers.Dropout(self.dropout_rate)
        self.mlp_dense2 = layers.Dense(self.embed_dim)
        self.drop2 = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, x, training=False):
        shortcut = x
        x = self.norm1(x)
        if self.shift and self.shift_size > 0:
            x = tf.roll(x, shift=[-self.shift_size, -self.shift_size], axis=[1, 2])
        windows = self.window_partition(x)
        win_seq = self.reshape_to_seq(windows)
        attn_out = self.attn(win_seq, win_seq, training=training)
        attn_out = self.drop1(attn_out, training=training)
        attn_out = self.reshape_to_2d(attn_out)
        x = self.window_reverse(attn_out)
        if self.shift and self.shift_size > 0:
            x = tf.roll(x, shift=[self.shift_size, self.shift_size], axis=[1, 2])
        x = x + shortcut
        shortcut2 = x
        x = self.norm2(x)
        x = self.mlp_dense1(x)
        x = self.mlp_drop(x, training=training)
        x = self.mlp_dense2(x)
        x = self.drop2(x, training=training)
        x = x + shortcut2
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'embed_dim': self.embed_dim, 'num_heads': self.num_heads,
            'window_size': self.window_size, 'h_patches': self.h_patches,
            'w_patches': self.w_patches, 'mlp_ratio': self.mlp_ratio,
            'dropout_rate': self.dropout_rate, 'shift': self.shift,
        })
        return config


class SwinStage(layers.Layer):
    def __init__(self, depth, embed_dim, num_heads, window_size,
                 h_patches, w_patches, mlp_ratio=2.0, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.depth = depth
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.h_patches = h_patches
        self.w_patches = w_patches
        self.mlp_ratio = mlp_ratio
        self.dropout_rate = dropout_rate

    def build(self, input_shape):
        self.blocks = []
        for i in range(self.depth):
            self.blocks.append(
                WindowAttentionBlock(
                    embed_dim=self.embed_dim,
                    num_heads=self.num_heads,
                    window_size=self.window_size,
                    h_patches=self.h_patches,
                    w_patches=self.w_patches,
                    mlp_ratio=self.mlp_ratio,
                    dropout_rate=self.dropout_rate,
                    shift=(i % 2 == 1),
                    name=f'swin_block_{i}'
                )
            )
        super().build(input_shape)

    def call(self, x, training=False):
        for block in self.blocks:
            x = block(x, training=training)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'depth': self.depth, 'embed_dim': self.embed_dim,
            'num_heads': self.num_heads, 'window_size': self.window_size,
            'h_patches': self.h_patches, 'w_patches': self.w_patches,
            'mlp_ratio': self.mlp_ratio, 'dropout_rate': self.dropout_rate,
        })
        return config


class CrossTemporalAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.dropout_rate = dropout_rate

    def build(self, input_shape):
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.embed_dim // self.num_heads,
            dropout=self.dropout_rate
        )
        self.drop = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, x, training=False):
        shape = tf.shape(x)
        B, T, H, W, C = shape[0], shape[1], shape[2], shape[3], shape[4]
        x_reshaped = tf.transpose(x, [0, 2, 3, 1, 4])
        x_reshaped = tf.reshape(x_reshaped, [B * H * W, T, C])
        x_norm = self.norm(x_reshaped)
        attn_out = self.attn(x_norm, x_norm, training=training)
        attn_out = self.drop(attn_out, training=training)
        x_reshaped = x_reshaped + attn_out
        x_out = tf.reshape(x_reshaped, [B, H, W, T, C])
        x_out = tf.transpose(x_out, [0, 3, 1, 2, 4])
        return x_out

    def get_config(self):
        config = super().get_config()
        config.update({
            'embed_dim': self.embed_dim,
            'num_heads': self.num_heads,
            'dropout_rate': self.dropout_rate,
        })
        return config


class TemporalConvBlock(layers.Layer):
    def __init__(self, channels, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv1D(
            self.channels, self.kernel_size, padding='same', activation='gelu'
        )
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        super().build(input_shape)

    def call(self, x, training=False):
        shape = tf.shape(x)
        B, T, H, W, C = shape[0], shape[1], shape[2], shape[3], shape[4]
        x_reshaped = tf.transpose(x, [0, 2, 3, 1, 4])
        x_reshaped = tf.reshape(x_reshaped, [B * H * W, T, C])
        out = self.conv(x_reshaped)
        out = self.norm(out)
        out = tf.reshape(out, [B, H, W, T, self.channels])
        out = tf.transpose(out, [0, 3, 1, 2, 4])
        return out

    def get_config(self):
        config = super().get_config()
        config.update({'channels': self.channels, 'kernel_size': self.kernel_size})
        return config


class TemporalAggregation(layers.Layer):
    def __init__(self, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    def build(self, input_shape):
        self.temporal_attn = layers.Dense(1, activation=None)
        super().build(input_shape)

    def call(self, x, training=False):
        scores = self.temporal_attn(x)
        scores = tf.nn.softmax(scores, axis=1)
        weighted = x * scores
        aggregated = tf.reduce_sum(weighted, axis=1)
        return aggregated

    def get_config(self):
        config = super().get_config()
        config.update({'embed_dim': self.embed_dim})
        return config


class FrameSliceLayer(layers.Layer):
    def __init__(self, frame_index, **kwargs):
        super().__init__(**kwargs)
        self.frame_index = frame_index

    def call(self, x):
        return x[:, self.frame_index, :, :, :]

    def get_config(self):
        config = super().get_config()
        config.update({'frame_index': self.frame_index})
        return config


class StackFramesLayer(layers.Layer):
    def call(self, inputs):
        return tf.stack(inputs, axis=1)

    def get_config(self):
        return super().get_config()


class ExpandDimsLayer(layers.Layer):
    def __init__(self, axis, **kwargs):
        super().__init__(**kwargs)
        self.axis = axis

    def call(self, x):
        return tf.expand_dims(x, axis=self.axis)

    def get_config(self):
        config = super().get_config()
        config.update({'axis': self.axis})
        return config


# ============================================================================
# MODEL DEFINITION: Swin ST-Transformer
# ============================================================================

def build_st_transformer_model(seq_len):
    """Build Swin ST-Transformer for spatiotemporal binary segmentation."""

    h0 = IMG_HEIGHT // PATCH_SIZE
    w0 = IMG_WIDTH  // PATCH_SIZE
    h1, w1 = h0 // 2, w0 // 2
    h2, w2 = h1 // 2, w1 // 2

    dim0 = EMBED_DIM
    dim1 = EMBED_DIM * 2
    dim2 = EMBED_DIM * 4

    input_shape = (seq_len, IMG_HEIGHT, IMG_WIDTH, 1)
    inputs = layers.Input(shape=input_shape, name='video_input')

    # ── Shared Spatial Encoder Components ──
    patch_embed = PatchEmbedSwin(
        patch_size=PATCH_SIZE, embed_dim=dim0, name='patch_embed'
    )
    swin_stage1 = SwinStage(
        depth=SWIN_DEPTHS[0], embed_dim=dim0, num_heads=SWIN_NUM_HEADS[0],
        window_size=WINDOW_SIZE, h_patches=h0, w_patches=w0,
        mlp_ratio=MLP_RATIO, dropout_rate=SPATIAL_DROPOUT, name='swin_stage1'
    )
    merge1 = PatchMerging(output_dim=dim1, name='patch_merge1')
    swin_stage2 = SwinStage(
        depth=SWIN_DEPTHS[1], embed_dim=dim1, num_heads=SWIN_NUM_HEADS[1],
        window_size=WINDOW_SIZE, h_patches=h1, w_patches=w1,
        mlp_ratio=MLP_RATIO, dropout_rate=SPATIAL_DROPOUT, name='swin_stage2'
    )
    merge2 = PatchMerging(output_dim=dim2, name='patch_merge2')
    swin_stage3 = SwinStage(
        depth=SWIN_DEPTHS[2], embed_dim=dim2, num_heads=SWIN_NUM_HEADS[2],
        window_size=WINDOW_SIZE, h_patches=h2, w_patches=w2,
        mlp_ratio=MLP_RATIO, dropout_rate=SPATIAL_DROPOUT, name='swin_stage3'
    )

    # ── Process each frame through spatial encoder ──
    bottleneck_features = []
    skip1_features = []
    skip2_features = []

    for t in range(seq_len):
        frame = FrameSliceLayer(frame_index=t, name=f'slice_frame_{t}')(inputs)
        x = patch_embed(frame)
        x = swin_stage1(x)
        s1 = x
        x = merge1(x)
        x = swin_stage2(x)
        s2 = x
        x = merge2(x)
        x = swin_stage3(x)
        bottleneck_features.append(x)
        skip1_features.append(s1)
        skip2_features.append(s2)

    # ── Stack bottleneck features ──
    temporal_features = StackFramesLayer(name='stack_bottleneck')(bottleneck_features)

    # ── Temporal Module ──
    temporal_features = TemporalConvBlock(
        channels=dim2, kernel_size=3, name='temporal_conv'
    )(temporal_features)

    for i in range(NUM_TEMPORAL_LAYERS):
        temporal_features = CrossTemporalAttention(
            embed_dim=dim2, num_heads=NUM_TEMPORAL_HEADS,
            dropout_rate=TEMPORAL_DROPOUT, name=f'cross_temporal_attn_{i}'
        )(temporal_features)

    aggregated = TemporalAggregation(
        embed_dim=dim2, name='temporal_aggregation'
    )(temporal_features)

    # ── Use last frame's skip connections for decoder ──
    last_skip1 = skip1_features[-1]
    last_skip2 = skip2_features[-1]

    # ── U-Net Decoder ──
    x = layers.Conv2DTranspose(
        DECODER_FILTERS[0], 3, strides=2, padding='same', activation='relu',
        name='dec_up1'
    )(aggregated)
    x = layers.BatchNormalization(name='dec_bn1')(x)
    x = layers.Concatenate(name='dec_skip2')([x, last_skip2])
    x = layers.Conv2D(DECODER_FILTERS[0], 3, padding='same', activation='relu',
                       name='dec_conv1a')(x)
    x = layers.Conv2D(DECODER_FILTERS[0], 3, padding='same', activation='relu',
                       name='dec_conv1b')(x)
    x = layers.Dropout(0.2, name='dec_drop1')(x)

    x = layers.Conv2DTranspose(
        DECODER_FILTERS[1], 3, strides=2, padding='same', activation='relu',
        name='dec_up2'
    )(x)
    x = layers.BatchNormalization(name='dec_bn2')(x)
    x = layers.Concatenate(name='dec_skip1')([x, last_skip1])
    x = layers.Conv2D(DECODER_FILTERS[1], 3, padding='same', activation='relu',
                       name='dec_conv2a')(x)
    x = layers.Conv2D(DECODER_FILTERS[1], 3, padding='same', activation='relu',
                       name='dec_conv2b')(x)
    x = layers.Dropout(0.2, name='dec_drop2')(x)

    x = layers.Conv2DTranspose(
        DECODER_FILTERS[2], 3, strides=2, padding='same', activation='relu',
        name='dec_up3'
    )(x)
    x = layers.BatchNormalization(name='dec_bn3')(x)
    x = layers.Conv2D(DECODER_FILTERS[2], 3, padding='same', activation='relu',
                       name='dec_conv3')(x)
    x = layers.Dropout(0.1, name='dec_drop3')(x)

    x = layers.Conv2DTranspose(
        DECODER_FILTERS[3], 3, strides=2, padding='same', activation='relu',
        name='dec_up4'
    )(x)
    x = layers.BatchNormalization(name='dec_bn4')(x)
    x = layers.Conv2D(DECODER_FILTERS[3], 3, padding='same', activation='relu',
                       name='dec_conv4')(x)

    outputs = layers.Conv2D(
        1, 1, padding='same', activation='sigmoid', name='output'
    )(x)

    model = models.Model(inputs, outputs,
                          name=f'SwinST_Transformer_seq{seq_len}')
    model.compile(
        optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
        loss=combined_loss,
        metrics=[dice_coefficient, iou_metric]
    )
    return model


# ============================================================================
# STRICT TEMPORAL SPLIT – TRIPLE-LAYER LEAK PREVENTION (YEARLY)
# ============================================================================

def verify_no_data_leak(train_input_indices, train_target_indices,
                        val_input_indices, val_target_indices,
                        test_input_indices, test_target_indices,
                        train_years, val_years, test_years,
                        seq_len, cutoff_year):
    errors = []

    # Layer 1: Target year overlap
    trainval_target_years = set(train_years.tolist()) | set(val_years.tolist())
    test_target_years = set(test_years.tolist())
    overlap_years = trainval_target_years & test_target_years
    if overlap_years:
        errors.append(f"LAYER-1 LEAK: Target year overlap → {sorted(overlap_years)}")

    # Layer 2: Shared image indices
    trainval_all_idx = set()
    for idx_list in train_input_indices + val_input_indices:
        trainval_all_idx.update(idx_list)
    trainval_all_idx.update(train_target_indices.tolist())
    trainval_all_idx.update(val_target_indices.tolist())

    test_all_idx = set()
    for idx_list in test_input_indices:
        test_all_idx.update(idx_list)
    test_all_idx.update(test_target_indices.tolist())

    overlap_idx = trainval_all_idx & test_all_idx
    if overlap_idx:
        errors.append(
            f"LAYER-2 LEAK: {len(overlap_idx)} shared image indices "
            f"between train/val and test → {sorted(overlap_idx)[:10]}..."
        )

    # Layer 3: Temporal ordering
    max_trainval_target = max(trainval_target_years) if trainval_target_years else -1
    min_test_target = min(test_target_years) if test_target_years else float('inf')
    if max_trainval_target >= min_test_target:
        errors.append(
            f"LAYER-3 LEAK: Latest train/val target "
            f"({max_trainval_target}) >= earliest test target "
            f"({min_test_target})"
        )

    if errors:
        for e in errors:
            print(f"  ✗ {e}")
        raise ValueError(f"DATA LEAK DETECTED ({len(errors)} violations)!")

    print(f"  ✓ Layer-1: No target year overlap")
    print(f"  ✓ Layer-2: No shared image indices "
          f"(train/val uses {len(trainval_all_idx)} imgs, "
          f"test uses {len(test_all_idx)} imgs)")
    print(f"  ✓ Layer-3: Temporal gap OK "
          f"(last train/val target={max_trainval_target}, "
          f"first test target={min_test_target})")


def prepare_split(X_all, y_all, target_years_all, input_years_all,
                  input_idx_all, target_idx_all,
                  cutoff_year, seq_len, all_years_list):
    max_input_year = np.array([iy.max() for iy in input_years_all])
    min_input_year = np.array([iy.min() for iy in input_years_all])

    train_mask = (target_years_all <= cutoff_year) & (max_input_year <= cutoff_year)
    test_mask_strict = (target_years_all > cutoff_year) & (min_input_year > cutoff_year)
    test_mask_standard = (target_years_all > cutoff_year)

    if test_mask_strict.sum() >= 1:
        test_mask = test_mask_strict
        gap_mode = "STRICT (all input years > cutoff)"
    else:
        test_mask = test_mask_standard
        gap_mode = "STANDARD (target > cutoff, input may overlap)"
        print(f"  ⚠ Strict gap yielded 0 test samples; using standard")

    X_train = X_all[train_mask]
    y_train = y_all[train_mask]
    ty_train = target_years_all[train_mask]
    train_input_idx = [input_idx_all[i] for i in range(len(train_mask)) if train_mask[i]]
    train_target_idx = target_idx_all[train_mask]

    X_test = X_all[test_mask]
    y_test = y_all[test_mask]
    ty_test = target_years_all[test_mask]
    test_input_idx = [input_idx_all[i] for i in range(len(test_mask)) if test_mask[i]]
    test_target_idx = target_idx_all[test_mask]

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError(
            f"Empty split! Train={len(X_train)}, Test={len(X_test)} "
            f"with cutoff={cutoff_year}, seq_len={seq_len}"
        )

    sort_idx = np.argsort(ty_train)
    X_train = X_train[sort_idx]
    y_train = y_train[sort_idx]
    ty_train = ty_train[sort_idx]
    train_input_idx = [train_input_idx[i] for i in sort_idx]
    train_target_idx = train_target_idx[sort_idx]

    split_idx = int(len(X_train) * 0.85)
    split_idx = max(1, min(split_idx, len(X_train) - 1))

    X_tr, y_tr, ty_tr = X_train[:split_idx], y_train[:split_idx], ty_train[:split_idx]
    tr_input_idx = train_input_idx[:split_idx]
    tr_target_idx = train_target_idx[:split_idx]

    X_val, y_val, ty_val = X_train[split_idx:], y_train[split_idx:], ty_train[split_idx:]
    val_input_idx = train_input_idx[split_idx:]
    val_target_idx = train_target_idx[split_idx:]

    if len(X_val) == 0:
        raise ValueError("Empty validation set!")

    print(f"\n  Gap mode: {gap_mode}")
    verify_no_data_leak(
        tr_input_idx, tr_target_idx, val_input_idx, val_target_idx,
        test_input_idx, test_target_idx, ty_tr, ty_val, ty_test,
        seq_len, cutoff_year
    )

    X_tr   = np.expand_dims(X_tr,   axis=-1)
    y_tr   = np.expand_dims(y_tr,   axis=-1)
    X_val  = np.expand_dims(X_val,  axis=-1)
    y_val  = np.expand_dims(y_val,  axis=-1)
    X_test = np.expand_dims(X_test, axis=-1)
    y_test = np.expand_dims(y_test, axis=-1)

    print(f"\n  Train : {len(X_tr):>4d} samples  "
          f"(years {int(ty_tr.min())}-{int(ty_tr.max())})")
    print(f"  Val   : {len(X_val):>4d} samples  "
          f"(years {int(ty_val.min())}-{int(ty_val.max())})")
    print(f"  Test  : {len(X_test):>4d} samples  "
          f"(years {int(ty_test.min())}-{int(ty_test.max())})")
    print(f"  Total images in dataset: {len(all_years_list)}")

    return X_tr, y_tr, X_val, y_val, X_test, y_test, ty_test


# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_model(model, X_test, y_test, target_years_test,
                   pixel_area_km2, model_name, setup_name):
    pred = model.predict(X_test, verbose=0)
    persistence_pred = X_test[:, -1, :, :, :]

    results = []
    for name, predictions in [('Persistence', persistence_pred),
                               (model_name, pred)]:
        for i in range(len(y_test)):
            yt = y_test[i, :, :, 0]
            yp = predictions[i, :, :, 0]
            year = target_years_test[i]

            iou = iou_np(yt, yp)
            dice = dice_coefficient_np(yt, yp)
            yt_flat = (yt.flatten() > 0.5).astype(int)
            yp_flat = (yp.flatten() > 0.5).astype(int)
            prec = precision_score(yt_flat, yp_flat, zero_division=0)
            rec  = recall_score(yt_flat, yp_flat, zero_division=0)
            area_diff = calculate_area_difference(yt, yp, pixel_area_km2)

            results.append({
                'Setup': setup_name, 'Model': name, 'Year': int(year),
                'IoU': iou, 'Dice': dice, 'Precision': prec,
                'Recall': rec, 'Area_Diff_km2': area_diff
            })

    return pd.DataFrame(results)


# ============================================================================
# DATA LOADING
# ============================================================================

print("=" * 80)
print("LOADING DATA (YEARLY — SWIN ST-TRANSFORMER — MULTI SEQ-LEN)")
print("=" * 80)

files = sorted(glob.glob(os.path.join(DATA_DIR, "*.tif")))
file_info = []
for filepath in files:
    filename = os.path.basename(filepath)
    match = re.search(r'(\d{4})', filename)
    if match:
        file_info.append({
            'filepath': filepath, 'filename': filename,
            'year': int(match.group(1))
        })

df_files = pd.DataFrame(file_info).sort_values('year').reset_index(drop=True)
print(f"Found {len(df_files)} files, years {df_files['year'].min()}-{df_files['year'].max()}")

# Check minimum required for the largest sequence length
max_seq_len = max(SEQUENCE_LENGTHS)
min_required = max_seq_len + PREDICTION_HORIZON
if len(df_files) < min_required:
    print(f"\n  ✗ FATAL: Need ≥ {min_required} images for max seq_len={max_seq_len}, "
          f"only {len(df_files)} found.")
    sys.exit(1)
else:
    print(f"  ✓ Sufficient data: {len(df_files)} ≥ {min_required} "
          f"(for max seq_len={max_seq_len})")

all_images, all_years = [], []
for idx, row in df_files.iterrows():
    img = load_and_preprocess_image(row['filepath'])
    all_images.append(img)
    all_years.append(row['year'])
    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx+1}/{len(df_files)} images...")

print(f"✓ Loaded {len(all_images)} images")

with rasterio.open(df_files.iloc[0]['filepath']) as src:
    bounds = src.bounds
    center_lat = (bounds.top + bounds.bottom) / 2
    if src.crs and src.crs.to_epsg() == 4326:
        meters_per_deg_lon = 111320 * np.cos(np.radians(center_lat))
        width_m  = (bounds.right - bounds.left) * meters_per_deg_lon
        height_m = (bounds.top - bounds.bottom) * 111320
        total_area_km2 = (width_m * height_m) / 1e6
    else:
        total_area_km2 = (
            (bounds.right - bounds.left) * (bounds.top - bounds.bottom)
        ) / 1e6

pixel_area_km2 = total_area_km2 / (IMG_HEIGHT * IMG_WIDTH)
print(f"✓ Pixel area: {pixel_area_km2:.8f} km²")

# ============================================================================
# ARCHITECTURE INFO
# ============================================================================

h0_info = IMG_HEIGHT // PATCH_SIZE
print(f"\n{'='*80}")
print(f"  SWIN ST-TRANSFORMER ARCHITECTURE SUMMARY")
print(f"{'='*80}")
print(f"  Image size          : {IMG_HEIGHT}×{IMG_WIDTH}")
print(f"  Patch size          : {PATCH_SIZE}×{PATCH_SIZE}")
print(f"  After patch embed   : {h0_info}×{h0_info}×{EMBED_DIM}")
print(f"  Window size         : {WINDOW_SIZE}×{WINDOW_SIZE}")
print(f"  Swin stages         : {len(SWIN_DEPTHS)} stages, "
      f"depths={SWIN_DEPTHS}, heads={SWIN_NUM_HEADS}")
print(f"  Stage dims          : {EMBED_DIM} → {EMBED_DIM*2} → {EMBED_DIM*4}")
print(f"  Bottleneck          : {h0_info//4}×{h0_info//4}×{EMBED_DIM*4}")
print(f"  Temporal module     : Conv1D + {NUM_TEMPORAL_LAYERS}× "
      f"CrossTemporalAttn + Aggregation")
print(f"  Decoder             : U-Net style with skip connections")
print(f"  Sequence lengths    : {SEQUENCE_LENGTHS}")
print(f"  Batch size          : {BATCH_SIZE}")
print(f"{'='*80}")

# ============================================================================
# CUSTOM OBJECTS FOR MODEL LOADING
# ============================================================================

CUSTOM_OBJECTS = {
    'combined_loss': combined_loss,
    'dice_coefficient': dice_coefficient,
    'iou_metric': iou_metric,
    'PatchEmbedSwin': PatchEmbedSwin,
    'PatchMerging': PatchMerging,
    'WindowPartition': WindowPartition,
    'WindowReverse': WindowReverse,
    'WindowAttentionBlock': WindowAttentionBlock,
    'SwinStage': SwinStage,
    'CrossTemporalAttention': CrossTemporalAttention,
    'TemporalConvBlock': TemporalConvBlock,
    'TemporalAggregation': TemporalAggregation,
    'FrameSliceLayer': FrameSliceLayer,
    'StackFramesLayer': StackFramesLayer,
    'ExpandDimsLayer': ExpandDimsLayer,
}

# ============================================================================
# MAIN TRAINING + EVALUATION LOOP (OVER MULTIPLE SEQUENCE LENGTHS)
# ============================================================================

all_results = []
summary_rows = []
best_configs = {}
training_histories = {}
first_model_printed = False

for seq_len in SEQUENCE_LENGTHS:

    print("\n" + "█" * 80)
    print(f"  SEQUENCE LENGTH = {seq_len}  (Swin ST-Transformer)")
    print("█" * 80)

    # ── Check this seq_len has enough data ──
    min_req = seq_len + PREDICTION_HORIZON
    if len(all_images) < min_req:
        print(f"  ⚠ Skipping seq_len={seq_len}: need ≥ {min_req} images, "
              f"only {len(all_images)} available.")
        continue

    # ── Create sequences FRESH for this seq_len ──
    (X_all_seq, y_all_seq, input_years_all, target_years_all,
     input_idx_all, target_idx_all) = \
        create_sequences_with_years(all_images, all_years, seq_len)

    print(f"✓ {len(X_all_seq)} sequences (seq_len={seq_len})  |  "
          f"target years {int(target_years_all.min())}-"
          f"{int(target_years_all.max())}")

    for setup_name, cfg in SETUP_CONFIGS.items():
        cutoff = cfg['cutoff_year']
        label = cfg['test_label']
        tag = f"swin_st_seq{seq_len}_{setup_name.replace(' ', '')}"

        print(f"\n{'─'*70}")
        print(f"  {setup_name} (cutoff={cutoff})  |  "
              f"SeqLen={seq_len}  |  Model: Swin ST-Transformer")
        print(f"{'─'*70}")

        try:
            X_tr, y_tr, X_val, y_val, X_test, y_test, ty_test = \
                prepare_split(
                    X_all_seq, y_all_seq, target_years_all,
                    input_years_all, input_idx_all, target_idx_all,
                    cutoff, seq_len, all_years
                )
        except ValueError as e:
            print(f"  ⚠ Skipping: {e}")
            continue

        # ── Build & train (fresh model for each seq_len + setup) ──
        K.clear_session()
        tf.random.set_seed(42)
        np.random.seed(42)

        model = build_st_transformer_model(seq_len)

        # Print summary only once (first model)
        if not first_model_printed:
            model.summary()
            first_model_printed = True

        print(f"\n  Training {tag}  (max {EPOCHS} epochs, "
              f"early-stop patience={EARLY_STOP_PATIENCE})\n")

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=create_callbacks(tag),
            verbose=0
        )
        training_histories[tag] = history

        stopped_epoch = len(history.history['loss'])
        best_val_loss = min(history.history['val_loss'])
        best_val_dice = max(history.history['val_dice_coefficient'])
        best_val_iou  = max(history.history['val_iou_metric'])

        print(f"\n  ✓ Stopped at epoch {stopped_epoch}/{EPOCHS}  |  "
              f"Best val_loss={best_val_loss:.4f}  "
              f"val_dice={best_val_dice:.4f}  "
              f"val_iou={best_val_iou:.4f}")

        # ── Evaluate ──
        model_label = f'SwinST(seq={seq_len})'
        df_res = evaluate_model(
            model, X_test, y_test, ty_test,
            pixel_area_km2, model_label,
            f'{setup_name} ({label})'
        )
        df_res['SeqLen'] = seq_len
        all_results.append(df_res)

        for mdl_name in df_res['Model'].unique():
            sub = df_res[df_res['Model'] == mdl_name]
            summary_rows.append({
                'SeqLen': seq_len,
                'Setup': setup_name,
                'Model': mdl_name,
                'Epochs_Run': stopped_epoch,
                'Best_Val_Loss': (
                    best_val_loss if mdl_name != 'Persistence' else np.nan
                ),
                'IoU': sub['IoU'].mean(),
                'Dice': sub['Dice'].mean(),
                'Precision': sub['Precision'].mean(),
                'Recall': sub['Recall'].mean(),
                'Area_Diff_km2': sub['Area_Diff_km2'].mean(),
                'Abs_Area_Diff_km2': sub['Area_Diff_km2'].abs().mean(),
            })

    # ── Free memory after each seq_len ──
    del X_all_seq, y_all_seq, input_years_all, target_years_all
    del input_idx_all, target_idx_all
    import gc
    gc.collect()

# ============================================================================
# AGGREGATE RESULTS
# ============================================================================

df_all = pd.concat(all_results, ignore_index=True)
df_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 100)
print(f"  SWIN ST-TRANSFORMER RESULTS  –  All Sequence Lengths  –  Mean Test Metrics")
print("=" * 100)
print(df_summary.round(4).to_string(index=False))

# ============================================================================
# IDENTIFY BEST SEQUENCE LENGTH PER SETUP
# ============================================================================

df_model_only = df_summary[df_summary['Model'].str.contains('SwinST')].copy()
df_persist_only = df_summary[df_summary['Model'] == 'Persistence'].copy()

print("\n" + "=" * 100)
print("  BEST SEQUENCE LENGTH PER SETUP  (by mean IoU on test)")
print("=" * 100)

for setup_name in SETUP_CONFIGS:
    sub = df_model_only[df_model_only['Setup'] == setup_name]
    if sub.empty:
        continue
    best_row = sub.loc[sub['IoU'].idxmax()]
    best_configs[setup_name] = int(best_row['SeqLen'])
    print(f"  {setup_name}: Best SeqLen = {int(best_row['SeqLen'])}  "
          f"(IoU={best_row['IoU']:.4f}, Dice={best_row['Dice']:.4f}, "
          f"Precision={best_row['Precision']:.4f}, Recall={best_row['Recall']:.4f})")

# ============================================================================
# SIDE-BY-SIDE: ACTUAL vs PREDICTED (FOR EACH SEQ_LEN × SETUP)
# ============================================================================

print("\n" + "=" * 100)
print("  SIDE-BY-SIDE  –  Actual vs Predicted  (Swin ST-Transformer)")
print("=" * 100)

for seq_len in SEQUENCE_LENGTHS:
    for setup_name, cfg in SETUP_CONFIGS.items():
        label = cfg['test_label']
        sub = df_all[
            (df_all['Setup'] == f'{setup_name} ({label})') &
            (df_all['SeqLen'] == seq_len)
        ].copy()

        if sub.empty:
            continue

        print(f"\n{'━'*90}")
        print(f"  {setup_name} | Sequence Length = {seq_len} | {label}")
        print(f"{'━'*90}")

        years = sorted(sub['Year'].unique())
        models_in_sub = sub['Model'].unique()

        rows = []
        for year in years:
            row = {'Year': int(year)}
            for mdl in models_in_sub:
                m = sub[(sub['Year'] == year) & (sub['Model'] == mdl)]
                if m.empty:
                    continue
                m = m.iloc[0]
                prefix = 'Pred' if 'SwinST' in mdl else 'Pers'
                row[f'{prefix}_IoU']       = round(m['IoU'], 4)
                row[f'{prefix}_Dice']      = round(m['Dice'], 4)
                row[f'{prefix}_Precision'] = round(m['Precision'], 4)
                row[f'{prefix}_Recall']    = round(m['Recall'], 4)
                row[f'{prefix}_ΔArea_km²'] = round(m['Area_Diff_km2'], 6)
            rows.append(row)

        df_side = pd.DataFrame(rows)

        desired_order = ['Year']
        for prefix in ['Pred', 'Pers']:
            for metric in ['IoU', 'Dice', 'Precision', 'Recall', 'ΔArea_km²']:
                col = f'{prefix}_{metric}'
                if col in df_side.columns:
                    desired_order.append(col)
        df_side = df_side[[c for c in desired_order if c in df_side.columns]]

        print(df_side.to_string(index=False))

        numeric_cols = [c for c in df_side.columns if c != 'Year']
        means = df_side[numeric_cols].mean()
        print(f"{'─'*90}")
        mean_str = "  MEAN |"
        for c in numeric_cols:
            mean_str += f"  {c}={means[c]:.4f}"
        print(mean_str)

# ============================================================================
# CROSS-SEQUENCE-LENGTH COMPARISON TABLE
# ============================================================================

print("\n" + "=" * 100)
print("  CROSS-SEQUENCE-LENGTH COMPARISON")
print("=" * 100)

if not df_model_only.empty:
    print("\n  ── Model Performance by Sequence Length ──\n")
    pivot_cols = ['SeqLen', 'Setup', 'IoU', 'Dice', 'Precision', 'Recall',
                  'Area_Diff_km2', 'Abs_Area_Diff_km2', 'Epochs_Run', 'Best_Val_Loss']
    print(df_model_only[pivot_cols].round(4).to_string(index=False))

    # Best seq_len per setup
    print("\n  ── Best Sequence Length per Setup (by IoU) ──\n")
    for setup_name in df_model_only['Setup'].unique():
        sub = df_model_only[df_model_only['Setup'] == setup_name]
        best_row = sub.loc[sub['IoU'].idxmax()]
        print(f"    {setup_name}: Best seq_len = {int(best_row['SeqLen'])}  "
              f"(IoU={best_row['IoU']:.4f}, Dice={best_row['Dice']:.4f}, "
              f"Precision={best_row['Precision']:.4f}, Recall={best_row['Recall']:.4f})")

    # Average across setups
    print("\n  ── Average Performance Across Setups ──\n")
    avg_by_seq = df_model_only.groupby('SeqLen')[
        ['IoU', 'Dice', 'Precision', 'Recall', 'Abs_Area_Diff_km2']
    ].mean()
    print(avg_by_seq.round(4).to_string())

    best_avg_seq = avg_by_seq['IoU'].idxmax()
    print(f"\n  ★ Overall best sequence length (avg IoU): {best_avg_seq}")

if not df_persist_only.empty:
    print("\n  ── Persistence Baseline by Sequence Length ──\n")
    persist_cols = ['SeqLen', 'Setup', 'IoU', 'Dice', 'Precision', 'Recall']
    print(df_persist_only[persist_cols].round(4).to_string(index=False))

# Improvement over persistence
print("\n  ── Improvement Over Persistence ──\n")
for seq_len in SEQUENCE_LENGTHS:
    for setup_name in df_summary['Setup'].unique():
        model_row = df_model_only[
            (df_model_only['SeqLen'] == seq_len) &
            (df_model_only['Setup'] == setup_name)
        ]
        persist_row = df_persist_only[
            (df_persist_only['SeqLen'] == seq_len) &
            (df_persist_only['Setup'] == setup_name)
        ]
        if model_row.empty or persist_row.empty:
            continue
        m = model_row.iloc[0]
        p = persist_row.iloc[0]
        iou_diff = m['IoU'] - p['IoU']
        dice_diff = m['Dice'] - p['Dice']
        sign_iou = "+" if iou_diff >= 0 else ""
        sign_dice = "+" if dice_diff >= 0 else ""
        print(f"    seq={seq_len}, {setup_name}: "
              f"ΔIoU={sign_iou}{iou_diff:.4f}  "
              f"ΔDice={sign_dice}{dice_diff:.4f}  "
              f"(Model: IoU={m['IoU']:.4f}, Persist: IoU={p['IoU']:.4f})")

# ============================================================================
# FULL SUMMARY TABLE
# ============================================================================

print("\n" + "=" * 100)
print("  FULL SUMMARY TABLE")
print("=" * 100)

display_cols = [
    'SeqLen', 'Setup', 'Model', 'Epochs_Run',
    'IoU', 'Dice', 'Precision', 'Recall',
    'Area_Diff_km2', 'Abs_Area_Diff_km2'
]
print(df_summary[display_cols].round(4).to_string(index=False))

# ============================================================================
# VISUALIZATION: TRAINING CURVES (ALL SEQ_LENS × SETUPS)
# ============================================================================

n_seq = len(SEQUENCE_LENGTHS)
n_setups = len(SETUP_CONFIGS)
total_rows = n_seq * n_setups

fig, axes = plt.subplots(total_rows, 3, figsize=(18, 5 * total_rows))
if total_rows == 1:
    axes = axes[np.newaxis, :]

row_idx = 0
for seq_len in SEQUENCE_LENGTHS:
    for setup_name, cfg in SETUP_CONFIGS.items():
        tag = f"swin_st_seq{seq_len}_{setup_name.replace(' ', '')}"
        hist = training_histories.get(tag)
        if hist is None:
            row_idx += 1
            continue

        ax = axes[row_idx, 0]
        ax.plot(hist.history['loss'], label='Train Loss')
        ax.plot(hist.history['val_loss'], label='Val Loss')
        ax.set_title(f'{setup_name} | seq={seq_len} | Loss')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.legend(); ax.grid(True)

        ax = axes[row_idx, 1]
        ax.plot(hist.history['dice_coefficient'], label='Train Dice')
        ax.plot(hist.history['val_dice_coefficient'], label='Val Dice')
        ax.set_title(f'{setup_name} | seq={seq_len} | Dice')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Dice')
        ax.legend(); ax.grid(True)

        ax = axes[row_idx, 2]
        ax.plot(hist.history['iou_metric'], label='Train IoU')
        ax.plot(hist.history['val_iou_metric'], label='Val IoU')
        ax.set_title(f'{setup_name} | seq={seq_len} | IoU')
        ax.set_xlabel('Epoch'); ax.set_ylabel('IoU')
        ax.legend(); ax.grid(True)

        row_idx += 1

plt.tight_layout()
plt.savefig('training_curves_swin_st_yearly_all_seqlens.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Training curves saved → training_curves_swin_st_yearly_all_seqlens.png")

# ============================================================================
# VISUALIZATION: COMPARATIVE TRAINING CURVES (OVERLAY SEQ_LENS)
# ============================================================================

for setup_name, cfg in SETUP_CONFIGS.items():
    fig, axes_comp = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{setup_name} | Sequence Length Comparison (Yearly)', fontsize=14,
                 fontweight='bold')

    for seq_len in SEQUENCE_LENGTHS:
        tag = f"swin_st_seq{seq_len}_{setup_name.replace(' ', '')}"
        hist = training_histories.get(tag)
        if hist is None:
            continue

        axes_comp[0].plot(hist.history['val_loss'],
                          label=f'seq={seq_len}', linewidth=1.5)
        axes_comp[1].plot(hist.history['val_dice_coefficient'],
                          label=f'seq={seq_len}', linewidth=1.5)
        axes_comp[2].plot(hist.history['val_iou_metric'],
                          label=f'seq={seq_len}', linewidth=1.5)

    for ax, metric_name in zip(axes_comp, ['Val Loss', 'Val Dice', 'Val IoU']):
        ax.set_title(metric_name)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric_name)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = f'training_comparison_{setup_name.replace(" ", "_")}_yearly_seqlens.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Comparative training curves saved → {fname}")

# ============================================================================
# VISUALIZATION: SAMPLE PREDICTIONS (ALL SEQ_LENS × SETUPS)
# ============================================================================

for seq_len in SEQUENCE_LENGTHS:
    for setup_name, cfg in SETUP_CONFIGS.items():
        cutoff = cfg['cutoff_year']
        label = cfg['test_label']
        tag = f"swin_st_seq{seq_len}_{setup_name.replace(' ', '')}"

        # Recreate test data for this seq_len
        (X_all_v, y_all_v, iy_all_v, ty_all_v,
         _, _) = create_sequences_with_years(
            all_images, all_years, seq_len
        )

        min_input_year_v = np.array([iy.min() for iy in iy_all_v])
        test_mask_strict = (ty_all_v > cutoff) & (min_input_year_v > cutoff)
        test_mask_std = (ty_all_v > cutoff)
        test_mask = test_mask_strict if test_mask_strict.sum() >= 1 else test_mask_std

        X_test_v  = np.expand_dims(X_all_v[test_mask], axis=-1)
        y_test_v  = y_all_v[test_mask]
        ty_test_v = ty_all_v[test_mask]

        ckpt = f'checkpoints/{tag}_best.keras'
        if os.path.exists(ckpt):
            try:
                best_model = tf.keras.models.load_model(
                    ckpt, custom_objects=CUSTOM_OBJECTS, safe_mode=False,
                )
            except TypeError:
                best_model = tf.keras.models.load_model(
                    ckpt, custom_objects=CUSTOM_OBJECTS,
                )
        else:
            print(f"  ⚠ Checkpoint not found: {ckpt}, skipping visualisation")
            continue

        preds = best_model.predict(X_test_v, verbose=0)
        n_show = min(6, len(y_test_v))
        if n_show == 0:
            print(f"  ⚠ No test samples for seq={seq_len}, {setup_name}")
            continue

        indices = np.linspace(0, len(y_test_v) - 1, n_show, dtype=int)

        fig, axes_v = plt.subplots(n_show, 3, figsize=(15, 4 * n_show))
        if n_show == 1:
            axes_v = axes_v[np.newaxis, :]

        for row, idx in enumerate(indices):
            actual    = y_test_v[idx]
            predicted = (preds[idx, :, :, 0] > 0.5).astype(np.float32)

            diff = np.zeros((*actual.shape, 3))
            diff[:, :, 1] = actual * (1 - predicted)      # Green = FN
            diff[:, :, 0] = predicted * (1 - actual)      # Red   = FP
            diff[:, :, 2] = actual * predicted             # Blue  = TP

            year_lbl = int(ty_test_v[idx])
            iou_val  = iou_np(actual, predicted)
            dice_val = dice_coefficient_np(actual, predicted)

            axes_v[row, 0].imshow(actual, cmap='gray')
            axes_v[row, 0].set_title(f'Actual – {year_lbl}', fontsize=11)
            axes_v[row, 0].axis('off')

            axes_v[row, 1].imshow(predicted, cmap='gray')
            axes_v[row, 1].set_title(
                f'Predicted – {year_lbl}\n'
                f'IoU={iou_val:.4f}  Dice={dice_val:.4f}', fontsize=11
            )
            axes_v[row, 1].axis('off')

            axes_v[row, 2].imshow(diff)
            axes_v[row, 2].set_title(
                f'Difference – {year_lbl}\n(R=FP, G=FN, B=TP)', fontsize=11
            )
            axes_v[row, 2].axis('off')

        fig.suptitle(
            f'{setup_name} | Swin-ST seq={seq_len} | '
            f'Actual vs Predicted (Yearly)',
            fontsize=14, fontweight='bold'
        )
        plt.tight_layout()
        fname = (f'predictions_swin_st_{setup_name.replace(" ", "_")}'
                 f'_seq{seq_len}_yearly.png')
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"✓ Prediction samples saved → {fname}")

        # Clean up
        del X_all_v, y_all_v, iy_all_v, ty_all_v, X_test_v, y_test_v
        del best_model, preds
        import gc
        gc.collect()

# ============================================================================
# VISUALIZATION: YEARLY TREND PLOTS (ALL SEQ_LENS × SETUPS)
# ============================================================================

print("\n" + "=" * 80)
print("  YEARLY TREND PLOTS (SWIN ST-TRANSFORMER — ALL SEQ LENS)")
print("=" * 80)

for seq_len in SEQUENCE_LENGTHS:
    for setup_name, cfg in SETUP_CONFIGS.items():
        label = cfg['test_label']
        sub = df_all[
            (df_all['Setup'] == f'{setup_name} ({label})') &
            (df_all['SeqLen'] == seq_len)
        ].copy()

        if sub.empty:
            continue

        model_data = sub[
            sub['Model'].str.contains('SwinST')
        ].sort_values('Year')
        persist_data = sub[
            sub['Model'] == 'Persistence'
        ].sort_values('Year')

        fig, axes_t = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(
            f'{setup_name} | Swin-ST seq={seq_len} | Yearly Test Metrics',
            fontsize=14, fontweight='bold'
        )

        for ax, metric in zip(
            axes_t.flat, ['IoU', 'Dice', 'Precision', 'Recall']
        ):
            x_labels = [int(y) for y in model_data['Year']]
            ax.plot(x_labels, model_data[metric].values,
                    'b-o', label=f'SwinST(seq={seq_len})', markersize=5)
            if len(persist_data) > 0:
                ax.plot(x_labels, persist_data[metric].values,
                        'r--s', label='Persistence', markersize=4, alpha=0.7)
            ax.set_title(metric, fontsize=12)
            ax.set_ylabel(metric)
            ax.set_xlabel('Year')
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        fname = (f'yearly_trends_swin_st_{setup_name.replace(" ", "_")}'
                 f'_seq{seq_len}.png')
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"✓ Yearly trend plot saved → {fname}")

# ============================================================================
# VISUALIZATION: CROSS-SEQ-LEN COMPARISON TREND PLOTS
# ============================================================================

print("\n" + "=" * 80)
print("  CROSS-SEQUENCE-LENGTH TREND COMPARISON (YEARLY)")
print("=" * 80)

colors_seq = {4: 'blue', 5: 'green', 6: 'red'}
markers_seq = {4: 'o', 5: 's', 6: '^'}

for setup_name, cfg in SETUP_CONFIGS.items():
    label = cfg['test_label']

    fig, axes_cross = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(
        f'{setup_name} | All Sequence Lengths | Yearly Test Metrics',
        fontsize=14, fontweight='bold'
    )

    has_data = False

    for seq_len in SEQUENCE_LENGTHS:
        sub = df_all[
            (df_all['Setup'] == f'{setup_name} ({label})') &
            (df_all['SeqLen'] == seq_len) &
            (df_all['Model'].str.contains('SwinST'))
        ].copy().sort_values('Year')

        if sub.empty:
            continue

        has_data = True
        color = colors_seq.get(seq_len, 'black')
        marker = markers_seq.get(seq_len, 'o')

        for ax, metric in zip(
            axes_cross.flat, ['IoU', 'Dice', 'Precision', 'Recall']
        ):
            x_labels = [int(y) for y in sub['Year']]
            ax.plot(x_labels, sub[metric].values,
                    color=color, marker=marker, linewidth=1.5,
                    label=f'seq={seq_len}', markersize=5)

    # Add persistence from any seq_len (it's the same baseline)
    for seq_len in SEQUENCE_LENGTHS:
        persist_sub = df_all[
            (df_all['Setup'] == f'{setup_name} ({label})') &
            (df_all['SeqLen'] == seq_len) &
            (df_all['Model'] == 'Persistence')
        ].copy().sort_values('Year')

        if not persist_sub.empty:
            for ax, metric in zip(
                axes_cross.flat, ['IoU', 'Dice', 'Precision', 'Recall']
            ):
                x_labels = [int(y) for y in persist_sub['Year']]
                ax.plot(x_labels, persist_sub[metric].values,
                        'k--', label=f'Persistence(seq={seq_len})',
                        markersize=3, alpha=0.4, linewidth=0.8)
            break  # Only plot persistence once

    if has_data:
        for ax, metric in zip(
            axes_cross.flat, ['IoU', 'Dice', 'Precision', 'Recall']
        ):
            ax.set_title(metric, fontsize=12)
            ax.set_ylabel(metric)
            ax.set_xlabel('Year')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
            ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        fname = (f'cross_seqlen_trends_{setup_name.replace(" ", "_")}_yearly.png')
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"✓ Cross-seq-len trend plot saved → {fname}")
    else:
        plt.close(fig)
        print(f"  ⚠ No data for {setup_name}, skipping cross-seq-len plot")

# ============================================================================
# VISUALIZATION: BAR CHART COMPARISON ACROSS SEQUENCE LENGTHS
# ============================================================================

print("\n" + "=" * 80)
print("  BAR CHART COMPARISON ACROSS SEQUENCE LENGTHS (YEARLY)")
print("=" * 80)

fig, axes = plt.subplots(1, len(SETUP_CONFIGS), figsize=(8 * len(SETUP_CONFIGS), 6))
if len(SETUP_CONFIGS) == 1:
    axes = [axes]

metrics_to_plot = ['IoU', 'Dice', 'Precision', 'Recall']

for ax_idx, (setup_name, cfg) in enumerate(SETUP_CONFIGS.items()):
    ax = axes[ax_idx]
    sub = df_model_only[df_model_only['Setup'] == setup_name]
    if sub.empty:
        continue

    x = np.arange(len(metrics_to_plot))
    width = 0.2
    for i, sl in enumerate(SEQUENCE_LENGTHS):
        vals = sub[sub['SeqLen'] == sl][metrics_to_plot].values
        if len(vals) == 0:
            continue
        vals = vals[0]
        offset = (i - len(SEQUENCE_LENGTHS) / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=f'seq={sl}')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8)

    best = best_configs.get(setup_name)
    ax.set_title(f'{setup_name} – Sequence Comparison (best={best})')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_to_plot)
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('sequence_comparison_bar_swin_st_yearly.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Sequence comparison chart saved → sequence_comparison_bar_swin_st_yearly.png")

# ============================================================================
# VISUALIZATION: SAMPLE PREDICTIONS (BEST CONFIG PER SETUP)
# ============================================================================

print("\n" + "=" * 80)
print("  BEST CONFIG PREDICTIONS (YEARLY)")
print("=" * 80)

for setup_name, cfg in SETUP_CONFIGS.items():
    best_seq = best_configs.get(setup_name)
    if best_seq is None:
        continue

    cutoff = cfg['cutoff_year']
    label = cfg['test_label']
    tag = f"swin_st_seq{best_seq}_{setup_name.replace(' ', '')}"

    (X_all_v, y_all_v, iy_all_v, ty_all_v,
     _, _) = create_sequences_with_years(
        all_images, all_years, best_seq
    )

    min_input_year_v = np.array([iy.min() for iy in iy_all_v])
    test_mask_strict = (ty_all_v > cutoff) & (min_input_year_v > cutoff)
    test_mask_std = (ty_all_v > cutoff)
    test_mask = test_mask_strict if test_mask_strict.sum() >= 1 else test_mask_std

    X_test_v  = np.expand_dims(X_all_v[test_mask], axis=-1)
    y_test_v  = y_all_v[test_mask]
    ty_test_v = ty_all_v[test_mask]

    ckpt = f'checkpoints/{tag}_best.keras'
    if os.path.exists(ckpt):
        try:
            best_model = tf.keras.models.load_model(
                ckpt, custom_objects=CUSTOM_OBJECTS, safe_mode=False,
            )
        except TypeError:
            best_model = tf.keras.models.load_model(
                ckpt, custom_objects=CUSTOM_OBJECTS,
            )
    else:
        print(f"  ⚠ Checkpoint not found: {ckpt}, skipping best-config visualisation")
        continue

    preds = best_model.predict(X_test_v, verbose=0)
    n_show = min(6, len(y_test_v))
    if n_show == 0:
        continue

    indices = np.linspace(0, len(y_test_v) - 1, n_show, dtype=int)

    fig, axes_v = plt.subplots(n_show, 3, figsize=(15, 4 * n_show))
    if n_show == 1:
        axes_v = axes_v[np.newaxis, :]

    for row, idx in enumerate(indices):
        actual    = y_test_v[idx]
        predicted = (preds[idx, :, :, 0] > 0.5).astype(np.float32)

        diff = np.zeros((*actual.shape, 3))
        diff[:, :, 1] = actual * (1 - predicted)
        diff[:, :, 0] = predicted * (1 - actual)
        diff[:, :, 2] = actual * predicted

        year_lbl = int(ty_test_v[idx])
        iou_val  = iou_np(actual, predicted)
        dice_val = dice_coefficient_np(actual, predicted)

        axes_v[row, 0].imshow(actual, cmap='gray')
        axes_v[row, 0].set_title(f'Actual – {year_lbl}', fontsize=11)
        axes_v[row, 0].axis('off')

        axes_v[row, 1].imshow(predicted, cmap='gray')
        axes_v[row, 1].set_title(
            f'Predicted – {year_lbl}\n'
            f'IoU={iou_val:.4f}  Dice={dice_val:.4f}', fontsize=11
        )
        axes_v[row, 1].axis('off')

        axes_v[row, 2].imshow(diff)
        axes_v[row, 2].set_title(
            f'Difference – {year_lbl}\n(R=FP, G=FN, B=TP)', fontsize=11
        )
        axes_v[row, 2].axis('off')

    fig.suptitle(
        f'{setup_name} | BEST Swin-ST seq={best_seq} | '
        f'Actual vs Predicted (Yearly)',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    fname = (f'best_predictions_swin_st_{setup_name.replace(" ", "_")}'
             f'_seq{best_seq}_yearly.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Best-config prediction samples saved → {fname}")

    del X_all_v, y_all_v, iy_all_v, ty_all_v, X_test_v, y_test_v
    del best_model, preds
    import gc
    gc.collect()

print("\n" + "=" * 100)
print("  ✓ ALL DONE – Swin ST-Transformer yearly multi-sequence evaluation complete")
print("=" * 100)

E0000 00:00:1777007931.590311      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777007931.661564      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777007932.199903      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777007932.199972      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777007932.199976      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777007932.199980      55 computation_placer.cc:177] computation placer already registered. Please check linka

✓ GPU configured: 2 GPU(s) available
LOADING DATA (YEARLY — SWIN ST-TRANSFORMER — MULTI SEQ-LEN)
Found 39 files, years 1987-2025
  ✓ Sufficient data: 39 ≥ 7 (for max seq_len=6)


I0000 00:00:1777007958.896353      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777007958.899152      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


  Processed 10/39 images...
  Processed 20/39 images...
  Processed 30/39 images...
✓ Loaded 39 images
✓ Pixel area: 0.32843740 km²

  SWIN ST-TRANSFORMER ARCHITECTURE SUMMARY
  Image size          : 256×256
  Patch size          : 4×4
  After patch embed   : 64×64×64
  Window size         : 8×8
  Swin stages         : 3 stages, depths=[2, 2, 2], heads=[2, 4, 8]
  Stage dims          : 64 → 128 → 256
  Bottleneck          : 16×16×256
  Temporal module     : Conv1D + 2× CrossTemporalAttn + Aggregation
  Decoder             : U-Net style with skip connections
  Sequence lengths    : [4, 5, 6]
  Batch size          : 2

████████████████████████████████████████████████████████████████████████████████
  SEQUENCE LENGTH = 4  (Swin ST-Transformer)
████████████████████████████████████████████████████████████████████████████████
✓ 35 sequences (seq_len=4)  |  target years 1991-2025

──────────────────────────────────────────────────────────────────────
  Setup 1 (cutoff=2015)  |  SeqLen=4  |  M

Model: "SwinST_Transformer_seq4"

...

 Trainable params: 3,234,642 (12.34 MB)

 Non-trainable params: 480 (1.88 KB)


  Training swin_st_seq4_Setup1  (max 200 epochs, early-stop patience=20)

──────────────────────────────────────────────────────────────────────────────────────
  Ep/Tot  │     Loss │    VLoss │   Dice │  VDice │    IoU │   VIoU │        LR │ Note
──────────────────────────────────────────────────────────────────────────────────────


I0000 00:00:1777008016.180312     126 service.cc:152] XLA service 0x7f1b9002b3d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777008016.180367     126 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777008016.180374     126 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777008028.503091     126 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777008132.815562     126 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   1/200  │   1.7490 │   1.6439 │ 0.0530 │ 0.0508 │ 0.0276 │ 0.0286 │  1.00e-04 │ ★ saved
   2/200  │   1.6587 │   1.6329 │ 0.0551 │ 0.0509 │ 0.0331 │ 0.0281 │  1.00e-04 │ ★ saved
   3/200  │   1.5847 │   1.6001 │ 0.0580 │ 0.0518 │ 0.0427 │ 0.0290 │  1.00e-04 │ ★ saved
   4/200  │   1.5218 │   1.5482 │ 0.0622 │ 0.0540 │ 0.0629 │ 0.0470 │  1.00e-04 │ ★ saved
   5/200  │   1.4655 │   1.4942 │ 0.0687 │ 0.0575 │ 0.0973 │ 0.0676 │  1.00e-04 │ ★ saved
   6/200  │   1.4177 │   1.4481 │ 0.0759 │ 0.0619 │ 0.1380 │ 0.0986 │  1.00e-04 │ ★ saved
   7/200  │   1.3774 │   1.4118 │ 0.0822 │ 0.0660 │ 0.1806 │ 0.2271 │  1.00e-04 │ ★ saved
   8/200  │   1.3419 │   1.3790 │ 0.0881 │ 0.0705 │ 0.2314 │ 0.3861 │  1.00e-04 │ ★ saved
   9/200  │   1.3103 │   1.3419 │ 0.0942 │ 0.0761 │ 0.2838 │ 0.4194 │  1.00e-04 │ ★ saved
  10/200  │   1.2813 │   1.3087 │ 0.1004 │ 0.0819 │ 0.3257 │ 0.4359 │  1.00e-04 │ ★ saved
  11/200  │   1.2539 │   1.2748 │ 0.1068 │ 0.0885 │ 0.3597 │ 0.4467 │  1.00e-04 │ ★ saved
  12/200  